# Banking Dataset Governance

Unity Catalog governance on the `Customer` table for two groups:

| Group | PII columns (phone, email, aadhaar) | Rows visible |
| --- | --- | --- |
| `de` | Full values | All rows |
| `da` | Masked (last 4 digits / domain only) | Only `active` customers |

Implemented with column masks + a row filter (SQL UDFs) applied via `ALTER TABLE`, plus `GRANT SELECT` for both groups.

In [0]:
%sql
-- 1. Governance functions: column masks (PII) + row filter
-- Data_engineer sees full values and all rows; everyone else (incl. da) gets masked values and active rows only.

-- column masking function
CREATE OR REPLACE FUNCTION dev.silver.mask_last4(val STRING)
RETURN CASE 
  WHEN is_account_group_member('de') THEN val
  ELSE concat('XXXXXX', right(val, 4)) 
END;

CREATE OR REPLACE FUNCTION dev.silver.mask_email(val STRING)
RETURN CASE WHEN is_account_group_member('de') THEN val
            ELSE concat('***@', split(val, '@')[1]) END;

-- row filter logic
CREATE OR REPLACE FUNCTION dev.silver.active_rows_only(status STRING)
RETURN is_account_group_member('de') OR status = 'active';

In [0]:
%sql
-- 3. Apply column masks, row filter, and grants

-- Row-level masking: only de sees dormant/inactive customers
ALTER TABLE dev.silver.Customer
    SET ROW FILTER dev.silver.active_rows_only ON (status);

-- Column-level masking: PII hidden from non-de users (incl. da)
ALTER TABLE dev.silver.Customer ALTER COLUMN phoneNumber   SET MASK dev.silver.mask_last4;
ALTER TABLE dev.silver.Customer ALTER COLUMN email         SET MASK dev.silver.mask_email;
ALTER TABLE dev.silver.Customer ALTER COLUMN aadharNumber  SET MASK dev.silver.mask_last4;

-- Grants: let both groups query the table
GRANT USE CATALOG ON CATALOG dev TO `de`;
GRANT USE CATALOG ON CATALOG dev TO `da`;
GRANT USE SCHEMA  ON SCHEMA  dev.silver TO `de`;
GRANT USE SCHEMA  ON SCHEMA  dev.silver TO `da`;
GRANT SELECT      ON TABLE   dev.silver.Customer TO `de`;
GRANT SELECT      ON TABLE   dev.silver.Customer TO `da`;

--you can revoke permissions as well

#Results

###DE SEE THIS

In [0]:
%sql
-- run as de
SELECT * FROM dev.silver.Customer ORDER BY id;

id,name,phoneNumber,email,aadharNumber,accBal,status
1,Rahul Sharma,9876543210,rahul.sharma@gmail.com,123456789012,150000.00,active
2,Priya Patel,9812345678,priya.patel@gmail.com,234567890123,82500.50,active
3,Amit Kumar,9898987654,amit.kumar@yahoo.com,345678901234,12750.25,dormant
4,Sneha Reddy,9765432109,sneha.reddy@gmail.com,456789012345,450000.00,active
5,Vikram Singh,9654321098,vikram.singh@outlook.com,567890123456,0.00,inactive
6,Anjali Gupta,9543210987,anjali.gupta@gmail.com,678901234567,67500.75,active
7,Rohan Mehta,9432109876,rohan.mehta@gmail.com,789012345678,23100.00,active
8,Kavya Nair,9321098765,kavya.nair@gmail.com,890123456789,98000.40,active
9,Arjun Verma,9210987654,arjun.verma@yahoo.com,901234567890,15250.60,dormant
10,Divya Joshi,9109876543,divya.joshi@gmail.com,012345678901,310500.00,active


###da view

![](/Workspace/Users/sakshijain7945@gmail.com/cyntexa/Emp/da.png)